# Chapter 9 §9.4: News Research Summarizer

Query decomposition, parallel ReAct research, and recursive summarization keep each model call focused instead of overflowing one context window.


In [ ]:
%pip install -r ../requirements.txt -q


## Runtime setup

Cells tagged `chapter-example` contain the chapter's core examples. Supporting cells provide imports, fixtures, or safe local stand-ins for external services.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

import dspy

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY in ../.env before running this notebook.")

lm = dspy.LM("openai/gpt-4o-mini", api_key=api_key)
dspy.configure(lm=lm)


## Local web-search stand-in

This example reuses Chapter 8's `web_search()` tool. The deterministic stand-in keeps the notebook self-contained; replace it with that live tool for current-news research.


In [ ]:
def web_search(query: str) -> str:
    """Return deterministic source snippets for the notebook demo."""
    return (
        f"Search results for {query}: "
        "Source A reports the event date as 2026-07-10. "
        "Source B confirms the date and adds implementation details. "
        "URLs: https://example.com/source-a, https://example.org/source-b"
    )


## Query decomposition and research


In [ ]:
class GenerateResearchQueries(dspy.Signature):
    """Break down a news topic into 3-5 specific,
    answerable research questions.

    Good queries are:
    - Specific and focused (not broad or vague)
    - Answerable with factual information
    - Non-overlapping (cover different aspects)
    - Ordered by importance (most critical first)
    """

    topic: str = dspy.InputField(
        desc="The breaking news topic to research"
    )
    queries: list[str] = dspy.OutputField(
        desc="List of 3-5 focused research questions"
    )


In [ ]:
class ResearchQuery(dspy.Signature):
    """Research a question using current web sources."""

    query: str = dspy.InputField(
        desc="The specific question to research"
    )
    findings: str = dspy.OutputField(
        desc="Factual findings supported by the search results"
    )
    sources: list[str] = dspy.OutputField(
        desc="URLs supporting the findings"
    )


## Parallel research and synthesis


In [ ]:

class SynthesizeNewsBrief(dspy.Signature):
    """Synthesize research findings into a concise news brief.

    Identify conflicts between sources. Prefer better-supported and
    more recent evidence when the findings justify that choice.
    Preserve unresolved disagreements instead of silently choosing
    one version.
    """

    topic: str = dspy.InputField(
        desc="The news topic being researched"
    )
    research_findings: str = dspy.InputField(
        desc="Findings from multiple research queries, including sources"
    )
    brief: str = dspy.OutputField(
        desc="A concise brief that reports facts and unresolved conflicts"
    )

class NewsResearchSummarizer(dspy.Module):
    def __init__(self):
        super().__init__()
        self.query_generator = dspy.Predict(GenerateResearchQueries)
        self.researcher = dspy.ReAct(
    ResearchQuery,
    tools=[web_search],
    max_iters=5,
   )

        self.synthesizer = dspy.ChainOfThought(SynthesizeNewsBrief)

    def forward(self, topic):
        # Stage 1: Generate research queries
        query_result = self.query_generator(topic=topic)
        queries = query_result.queries

        # Stage 2: Research each query in parallel
        parallel = dspy.Parallel(num_threads=len(queries))
        exec_pairs = [
            (self.researcher, {"query": q}) for q in queries
        ]
        results = parallel(exec_pairs)
        findings = [
    (
        f"Q: {q}\n"
        f"A: {r.findings}\n"
        f"Sources: {', '.join(r.sources)}"
    )
    for q, r in zip(queries, results)
]

        # Stage 3: Synthesize into news brief
        all_findings = "\n\n".join(findings)
        brief_result = self.synthesizer(
            topic=topic,
            research_findings=all_findings
        )

        return dspy.Prediction(
            queries=queries,
            findings=findings,
            brief=brief_result.brief
        )


## Recursive summarization

Pass tokenizer-aware, overlapping chunks from the ingestion layer into this module.


In [ ]:
class SummarizeChunk(dspy.Signature):
    """Summarize a text chunk, preserving key facts and sources."""

    chunk: str = dspy.InputField(desc="Text chunk to summarize")
    topic: str = dspy.InputField(desc="Original research topic")
    summary: str = dspy.OutputField(
        desc="Concise summary preserving key facts"
    )

class RecursiveSummarizer(dspy.Module):
    def __init__(self):
        super().__init__()
        self.chunk_summarizer = dspy.Predict(SummarizeChunk)
        self.synthesizer = dspy.ChainOfThought(SynthesizeNewsBrief)

    def forward(self, topic, chunks):
        # Summarize each chunk
        summaries = []
        for chunk in chunks:
            result = self.chunk_summarizer(
                chunk=chunk, topic=topic
            )
            summaries.append(result.summary)

        # Synthesize summaries into final brief
        combined = "\n\n".join(summaries)
        return self.synthesizer(
            topic=topic,
            research_findings=combined
        )
